---
## VER Human+Object Preparation Pipeline (Dual-Class)

The following cells prepare a combined dataset of **human** and **object** signals for downstream ML models.

### Step 1 – Import Libraries

Import all libraries needed to load, combine, and prepare the dual-class dataset.

In [1]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.ensemble import RandomForestClassifier

import matplotlib.pyplot as plt

# Import TensorFlow/Keras lazily so the notebook still works if TF is not installed elsewhere
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Optional engine for reading .xlsx header schema (if needed)
try:
    import openpyxl  # noqa: F401
except ImportError:
    openpyxl = None

# Directory for saving result figures
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

print("Libraries for VER dual-class pipeline imported.")

Libraries for VER dual-class pipeline imported.


### Step 2 – Load Datasets (Human & Object)

Load the separate human and object CSV files referenced in the project into individual DataFrames.

In [2]:
# Paths to the original human and object datasets (relative to project root)
human_path = "Dataset/1_human_5000_kashif_24_01.csv"
object_path = "Dataset/1_box_signal_5000_24_01.csv"

human_df = pd.read_csv(human_path, header=None)
object_df = pd.read_csv(object_path, header=None)

print("Human dataset shape:", human_df.shape)
print("Object dataset shape:", object_df.shape)

Human dataset shape: (5000, 25017)
Object dataset shape: (5000, 25017)


### Step 3 – Combine Datasets in Memory

Concatenate the human and object DataFrames into a single combined DataFrame **without writing anything to disk**.

In [3]:
combined_df = pd.concat([human_df, object_df], ignore_index=True)

print("Combined DataFrame shape:", combined_df.shape)
print("Number of human rows:", len(human_df))
print("Number of object rows:", len(object_df))

Combined DataFrame shape: (10000, 25017)
Number of human rows: 5000
Number of object rows: 5000


### Step 4 – Apply Header Structure

Load the header schema from `Project_Docs/Header structure Winter 2021 (1).xlsx` and apply those column names/structure to the combined DataFrame (in memory only).

In [4]:
header_path = os.path.join("Project_Docs", "Header structure Winter 2021 (1).xlsx")

# Load header schema (assume first row contains the desired column names)
header_schema = pd.read_excel(header_path, header=None)
header_columns = header_schema.iloc[0].tolist()

# Apply header names ONLY to the first N columns,
# keeping all remaining columns and data in combined_df unchanged
n_header_cols = len(header_columns)

# We can only apply as many headers as there are columns in combined_df
n_apply = min(n_header_cols, combined_df.shape[1])

current_cols = list(combined_df.columns)
for i in range(n_apply):
    current_cols[i] = header_columns[i]
combined_df.columns = current_cols

print(f"Header schema columns: {n_header_cols}")
print(f"Applied header names to first {n_apply} columns. Total columns in combined_df remain: {combined_df.shape[1]}")

Header schema columns: 17
Applied header names to first 17 columns. Total columns in combined_df remain: 25017


### Step 5 – Update Column 3 for Dual Classes

Modify **column 3** so it encodes the two classes (human vs. object) for each row in the combined DataFrame.

In [5]:
# Ensure there are at least 3 columns to hold the class labels
if combined_df.shape[1] < 3:
    raise ValueError("Combined DataFrame has fewer than 3 columns; cannot set class labels in column 3.")

n_human = len(human_df)
n_object = len(object_df)

# Use position-based indexing: third column (index 2) holds the dual-class label (0/1 for ML models)
# Convention: 0 = object, 1 = human
combined_df.iloc[:n_human, 2] = 1  # human
combined_df.iloc[n_human:n_human + n_object, 2] = 0  # object

print("Column 3 updated with numeric labels: 1=human, 0=object.")
print(combined_df.shape)

Column 3 updated with numeric labels: 1=human, 0=object.
(10000, 25017)


### Step 6 – Preview Data for Modeling

Display the first 10 rows and first 20 columns of the final combined DataFrame to verify it is ready for ML models.

In [6]:
# Show first 10 rows and first 20 columns of the prepared dataset
combined_df.iloc[:10, :20]

,header length (number of table elements),data length,class detected,measurement type\nFFT / ADC,dependent on table element D either frequency resolution Δf or sampling time Δt,currently irrelevant,sampling frequency f_s (Hz),ADC resolution (bits),currently irrelevant,currently irrelevant,\ndistance between sensor and first object (round-trip-time in µs),FFT Window length,currently irrelevant,software version (RP),aux 1,aux 2,data …..,17,18,19
0,68.0,50000.0,1.0,1.0,512.0,1.0,1953125.0,14.0,15.0,2622.0,0.000000,0.0,0.0,2.0,1.0,0.0,260611.375000,-35.0,-31.0,-26.0
1,68.0,50000.0,1.0,1.0,512.0,1.0,1953125.0,14.0,15.0,2622.0,0.000000,0.0,0.0,2.0,1.0,0.0,260672.250000,-120.0,-133.0,-142.0
2,68.0,50000.0,1.0,1.0,512.0,1.0,1953125.0,14.0,15.0,2622.0,0.000000,0.0,0.0,2.0,1.0,0.0,260734.640625,-46.0,-47.0,-47.0
3,68.0,50000.0,1.0,1.0,512.0,1.0,1953125.0,14.0,15.0,2622.0,2.265292,0.0,0.0,2.0,1.0,0.0,260794.250000,111.0,101.0,67.0
4,68.0,50000.0,1.0,1.0,512.0,1.0,1953125.0,14.0,15.0,2622.0,0.000000,0.0,0.0,2.0,1.0,0.0,260860.000000,-99.0,-101.0,-102.0
5,68.0,50000.0,1.0,1.0,512.0,1.0,1953125.0,14.0,15.0,2622.0,0.000000,0.0,0.0,2.0,1.0,0.0,260926.796875,-18.0,-10.0,-7.0
6,68.0,50000.0,1.0,1.0,512.0,1.0,1953125.0,14.0,15.0,2622.0,0.000000,0.0,0.0,2.0,1.0,0.0,261006.359375,-118.0,-112.0,-111.0
7,68.0,50000.0,1.0,1.0,512.0,1.0,1953125.0,14.0,15.0,2622.0,0.159760,0.0,0.0,2.0,1.0,0.0,261068.390625,-79.0,-78.0,-75.0
8,68.0,50000.0,1.0,1.0,512.0,1.0,1953125.0,14.0,15.0,2622.0,0.000000,0.0,0.0,2.0,1.0,0.0,261147.593750,-20.0,-22.0,-29.0
9,68.0,50000.0,1.0,1.0,512.0,1.0,1953125.0,14.0,15.0,2622.0,0.000000,0.0,0.0,2.0,1.0,0.0,261221.062500,-26.0,-24.0,-21.0


### Step 7 – Drop Irrelevant Columns

From the combined DataFrame, drop columns with original positional indices **0**, **1**, and **3–17** (inclusive). The remaining columns will be used for modeling (label + features).

In [7]:
# Drop columns by original positional indices: 0, 1, and 3–17 (inclusive)
cols_to_drop_pos = [0, 1] + list(range(3, 18))

# Guard against cases where the DataFrame has fewer columns than the largest index
max_idx = combined_df.shape[1] - 1
cols_to_drop_pos = [idx for idx in cols_to_drop_pos if idx <= max_idx]

cols_to_drop = combined_df.columns[cols_to_drop_pos]

df_model = combined_df.drop(columns=cols_to_drop).copy()

print("Original combined_df shape:", combined_df.shape)
print("Columns dropped (by name):", list(cols_to_drop))
print("Modeling DataFrame shape (df_model):", df_model.shape)

Original combined_df shape: (10000, 25017)
Columns dropped (by name): ['header length (number of table elements)', 'data length', 'measurement type\nFFT / ADC', 'dependent on table element D either frequency resolution Δf  or sampling time Δt', 'currently irrelevant', 'sampling frequency f_s (Hz)', 'ADC resolution (bits) ', 'currently irrelevant', 'currently irrelevant', '\ndistance between sensor and first object (round-trip-time in µs)', 'FFT Window length', 'currently irrelevant', 'software version (RP)', 'aux 1', 'aux 2', 'data …..', 17]
Modeling DataFrame shape (df_model): (10000, 25000)


### Step 8 – Prepare Features (X) and Labels (y)

Use **column 3 (index 2)** as the dual-class label (0 = object, 1 = human) and all remaining columns in `df_model` as features.

In [8]:
# Column index for the dual-class label (from previous step)
label_col_idx = 2  # 0 = object, 1 = human

# Extract labels as a 1D numpy array of ints
y = df_model.iloc[:, label_col_idx].astype(int).values

# Features are all remaining columns in df_model except the label column
X = df_model.drop(df_model.columns[label_col_idx], axis=1).values.astype(np.float32)

print("Feature matrix X shape:", X.shape)
print("Label vector y shape:", y.shape)
print("Class distribution (value counts):")
print(pd.Series(y).value_counts())

Feature matrix X shape: (10000, 24999)
Label vector y shape: (10000,)
Class distribution (value counts):
-69      156
-76      148
-67      145
-68      143
-73      142
        ... 
-397       1
 535       1
 1643      1
-188       1
 6994      1
Name: count, Length: 978, dtype: int64


### Step 9 – Train/Test Split and Scaling

Split the data into **80% train / 20% test** with a fixed random state and standardize the features.

In [9]:


# Decide whether we can safely use stratification
class_counts = pd.Series(y).value_counts()
min_class_count = class_counts.min()

if min_class_count < 2:
    print(
        "Warning: At least one class has fewer than 2 samples. "
        "Proceeding without stratification in train_test_split."
    )
    stratify_arg = None
else:
    stratify_arg = y

# 80/20 train-test split with fixed random_state
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=stratify_arg,
)

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Class counts in y:")
print(class_counts)
print("\nX_train shape:", X_train_scaled.shape)
print("X_test shape:", X_test_scaled.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

# Dictionaries to collect model metrics for final comparison
model_accuracies = {}
model_macro_f1 = {}

Class counts in y:
-69      156
-76      148
-67      145
-68      143
-73      142
        ... 
-397       1
 535       1
 1643      1
-188       1
 6994      1
Name: count, Length: 978, dtype: int64

X_train shape: (8000, 24999)
X_test shape: (2000, 24999)
y_train shape: (8000,)
y_test shape: (2000,)


### SVM – Dual-Class Classification

Train an SVM classifier (`sklearn.svm.SVC`) on the scaled training data and evaluate on the test set.

In [ ]:
from sklearn.svm import LinearSVC
svm_clf = LinearSVC(random_state=42)
svm_clf.fit(X_train_scaled, y_train)
y_pred_svm = svm_clf.predict(X_test_scaled)
svm_accuracy = accuracy_score(y_test, y_pred_svm)
svm_macro_f1 = f1_score(y_test, y_pred_svm, average="macro")

print(f"SVM Test accuracy: {svm_accuracy:.4f}")
print("\nSVM Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_svm))
print("\nSVM Classification Report:\n")
print(classification_report(y_test, y_pred_svm, digits=4))

model_accuracies["SVM"] = svm_accuracy
model_macro_f1["SVM"] = svm_macro_f1

### Random Forest – Dual-Class Classification

Train a `RandomForestClassifier` on the scaled training data and evaluate on the test set.

In [ ]:
rf_clf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    random_state=42,
    n_jobs=-1,
)
rf_clf.fit(X_train_scaled, y_train)

y_pred_rf = rf_clf.predict(X_test_scaled)
rf_accuracy = accuracy_score(y_test, y_pred_rf)
rf_macro_f1 = f1_score(y_test, y_pred_rf, average="macro")

print(f"Random Forest Test accuracy: {rf_accuracy:.4f}")
print("\nRandom Forest Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))
print("\nRandom Forest Classification Report:\n")
print(classification_report(y_test, y_pred_rf, digits=4))

model_accuracies["Random Forest"] = rf_accuracy
model_macro_f1["Random Forest"] = rf_macro_f1

### CNN – Dual-Class Classification (TensorFlow/Keras)

Train a 1D CNN on the scaled feature sequences using `tensorflow.keras` and evaluate on the test set.

In [ ]:
# Prepare 3D input tensors for Conv1D: (samples, timesteps/features, channels)
X_train_cnn = X_train_scaled.reshape((X_train_scaled.shape[0], X_train_scaled.shape[1], 1))
X_test_cnn = X_test_scaled.reshape((X_test_scaled.shape[0], X_test_scaled.shape[1], 1))

cnn_model = keras.Sequential([
    layers.Input(shape=(X_train_cnn.shape[1], 1)),
    layers.Conv1D(32, kernel_size=3, activation="relu"),
    layers.Conv1D(64, kernel_size=3, activation="relu"),
    layers.GlobalAveragePooling1D(),
    layers.Dense(32, activation="relu"),
    layers.Dense(1, activation="sigmoid"),  # binary classification (0/1)
])

cnn_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

history_cnn = cnn_model.fit(
    X_train_cnn,
    y_train,
    epochs=5,
    batch_size=32,
    validation_split=0.2,
    verbose=1,
)

cnn_loss, cnn_acc = cnn_model.evaluate(X_test_cnn, y_test, verbose=0)
y_pred_cnn = (cnn_model.predict(X_test_cnn, verbose=0) > 0.5).astype(int).flatten()
cnn_macro_f1 = f1_score(y_test, y_pred_cnn, average="macro")

print(f"CNN Test accuracy: {cnn_acc:.4f}")
print("\nCNN Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_cnn))
print("\nCNN Classification Report:\n")
print(classification_report(y_test, y_pred_cnn, digits=4))

model_accuracies["CNN"] = float(cnn_acc)
model_macro_f1["CNN"] = cnn_macro_f1

### TCN – Dual-Class Classification (Dilated Causal Conv1D)

Train a simple Temporal Convolutional Network (TCN)-style model using stacked **dilated causal Conv1D** layers in Keras and evaluate on the test set.

In [ ]:
# Reuse the same 3D representation of the inputs (tf/keras already imported in CNN cell)
X_train_tcn = X_train_scaled.reshape((X_train_scaled.shape[0], X_train_scaled.shape[1], 1))
X_test_tcn = X_test_scaled.reshape((X_test_scaled.shape[0], X_test_scaled.shape[1], 1))

inputs = keras.Input(shape=(X_train_tcn.shape[1], 1))
x = inputs

# Stack of dilated causal Conv1D layers (TCN-style)
for dilation_rate in [1, 2, 4, 8]:
    x = layers.Conv1D(
        filters=32,
        kernel_size=3,
        padding="causal",
        activation="relu",
        dilation_rate=dilation_rate,
    )(x)

x = layers.GlobalAveragePooling1D()(x)
x = layers.Dense(32, activation="relu")(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

tcn_model = keras.Model(inputs=inputs, outputs=outputs)

tcn_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

history_tcn = tcn_model.fit(
    X_train_tcn,
    y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.2,
    verbose=1,
)

tcn_loss, tcn_acc = tcn_model.evaluate(X_test_tcn, y_test, verbose=0)
y_pred_tcn = (tcn_model.predict(X_test_tcn, verbose=0) > 0.5).astype(int).flatten()
tcn_macro_f1 = f1_score(y_test, y_pred_tcn, average="macro")

print(f"TCN Test accuracy: {tcn_acc:.4f}")
print("\nTCN Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_tcn))
print("\nTCN Classification Report:\n")
print(classification_report(y_test, y_pred_tcn, digits=4))

model_accuracies["TCN"] = float(tcn_acc)
model_macro_f1["TCN"] = tcn_macro_f1

### Conv-Transformer Hybrid – Dual-Class Classification

Hybrid model: Conv1D layers for local features, then Multi-Head Self-Attention for sequence context, then global pool and classifier.

In [ ]:
# Same 3D input as CNN/TCN
X_train_hybrid = X_train_scaled.reshape((X_train_scaled.shape[0], X_train_scaled.shape[1], 1))
X_test_hybrid = X_test_scaled.reshape((X_test_scaled.shape[0], X_test_scaled.shape[1], 1))

seq_len = X_train_hybrid.shape[1]
d_model = 64

inputs = keras.Input(shape=(seq_len, 1))
x = layers.Conv1D(32, kernel_size=3, activation="relu")(inputs)
x = layers.Conv1D(d_model, kernel_size=3, activation="relu")(x)
x = layers.MultiHeadAttention(num_heads=4, key_dim=d_model // 4)(x, x)
x = layers.GlobalAveragePooling1D()(x)
x = layers.Dense(32, activation="relu")(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

conv_transformer_model = keras.Model(inputs=inputs, outputs=outputs)
conv_transformer_model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

conv_transformer_model.fit(
    X_train_hybrid, y_train,
    epochs=5, batch_size=64, validation_split=0.2, verbose=1,
)

_, ct_acc = conv_transformer_model.evaluate(X_test_hybrid, y_test, verbose=0)
y_pred_ct = (conv_transformer_model.predict(X_test_hybrid, verbose=0) > 0.5).astype(int).flatten()
ct_macro_f1 = f1_score(y_test, y_pred_ct, average="macro")

print(f"Conv-Transformer Test accuracy: {ct_acc:.4f}")
print("\nConv-Transformer Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_ct))
print("\nConv-Transformer Classification Report:\n")
print(classification_report(y_test, y_pred_ct, digits=4))

model_accuracies["Conv-Transformer"] = float(ct_acc)
model_macro_f1["Conv-Transformer"] = ct_macro_f1

### Time Series Transformer – Dual-Class Classification

Transformer for time series: patch-style projection, positional encoding, transformer encoder block(s), then pool and classify.

In [ ]:
# Reuse 3D input
X_train_ts = X_train_scaled.reshape((X_train_scaled.shape[0], X_train_scaled.shape[1], 1))
X_test_ts = X_test_scaled.reshape((X_test_scaled.shape[0], X_test_scaled.shape[1], 1))

seq_len = X_train_ts.shape[1]
d_model = 64
num_heads = 4
ff_dim = 64

inputs = keras.Input(shape=(seq_len, 1))
x = layers.Conv1D(d_model, kernel_size=1)(inputs)

# Positional encoding (learned); expand for broadcasting (batch, seq_len, d_model)
pos_embed = layers.Embedding(input_dim=seq_len, output_dim=d_model)(tf.range(seq_len))
pos_embed = tf.expand_dims(pos_embed, 0)
x = x + pos_embed

# Transformer encoder: MultiHeadAttention + FFN
attn_out = layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model // num_heads)(x, x)
x = layers.LayerNormalization(epsilon=1e-6)(x + attn_out)
ffn = layers.Dense(ff_dim, activation="relu")(x)
ffn = layers.Dense(d_model)(ffn)
x = layers.LayerNormalization(epsilon=1e-6)(x + ffn)

x = layers.GlobalAveragePooling1D()(x)
x = layers.Dense(32, activation="relu")(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

ts_transformer_model = keras.Model(inputs=inputs, outputs=outputs)
ts_transformer_model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

ts_transformer_model.fit(
    X_train_ts, y_train,
    epochs=10, batch_size=32, validation_split=0.2, verbose=1,
)

_, ts_acc = ts_transformer_model.evaluate(X_test_ts, y_test, verbose=0)
y_pred_ts = (ts_transformer_model.predict(X_test_ts, verbose=0) > 0.5).astype(int).flatten()
ts_macro_f1 = f1_score(y_test, y_pred_ts, average="macro")

print(f"Time Series Transformer Test accuracy: {ts_acc:.4f}")
print("\nTime Series Transformer Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_ts))
print("\nTime Series Transformer Classification Report:\n")
print(classification_report(y_test, y_pred_ts, digits=4))

model_accuracies["Time Series Transformer"] = float(ts_acc)
model_macro_f1["Time Series Transformer"] = ts_macro_f1

### Model Comparison Summary

Summary table (Accuracy + Macro F1), bar chart, and best model by accuracy and by macro F1 for all six models.

In [ ]:
# Build summary DataFrame with Accuracy and Macro F1
summary_rows = [
    {"Model": name, "Accuracy": model_accuracies[name], "Macro F1": model_macro_f1[name]}
    for name in model_accuracies
]
summary_df = pd.DataFrame(summary_rows)

print("Model Summary (Accuracy & Macro F1):")
display(summary_df)

# Best model by accuracy and by macro F1
best_by_acc = summary_df.loc[summary_df["Accuracy"].idxmax(), "Model"]
best_by_f1 = summary_df.loc[summary_df["Macro F1"].idxmax(), "Model"]
print(f"\nBest model by Accuracy: {best_by_acc}")
print(f"Best model by Macro F1: {best_by_f1}")

# Bar chart: Accuracy and Macro F1 per model
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

x = np.arange(len(summary_df))
w = 0.35

axes[0].bar(x - w/2, summary_df["Accuracy"], width=w, label="Accuracy", color="steelblue")
axes[0].set_xticks(x)
axes[0].set_xticklabels(summary_df["Model"], rotation=45, ha="right")
axes[0].set_ylabel("Score")
axes[0].set_title("Accuracy by Model")
axes[0].legend()

axes[1].bar(x - w/2, summary_df["Macro F1"], width=w, label="Macro F1", color="coral")
axes[1].set_xticks(x)
axes[1].set_xticklabels(summary_df["Model"], rotation=45, ha="right")
axes[1].set_ylabel("Score")
axes[1].set_title("Macro F1 by Model")
axes[1].legend()

plt.tight_layout()

# Save the summary figure to the results directory
fig_path = RESULTS_DIR / "model_comparison_summary.png"
fig.savefig(fig_path, dpi=300, bbox_inches="tight")

plt.show()
print(f"Summary figure saved to: {fig_path}")